# RuleForge BERT Baseline (Colab)

This notebook fine-tunes `bert-base-chinese` on ToxiCN and produces test predictions.

## Setup

1. **Runtime > Change runtime type > T4 GPU** (or A100 if available)
2. Replace `REPO_URL` below with your actual repo URL (GitHub / Gitee / local zip)
3. Run all cells top to bottom
4. Download `bert_artifacts.tar.gz` at the end

## 0. Config — edit this cell

In [ ]:
# ===== EDIT HERE =====
# Option A: git clone (public/private repo)
REPO_URL = "https://github.com/cuizhenzhi/rule-forge.git"

# Option B: upload a zip instead (set REPO_URL = "" and upload manually)
# After upload, unzip into /content/rule-forge

BRANCH = "main"  # branch to clone
# =====================

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo & install deps

In [ ]:
import os

WORK_DIR = "/content/rule-forge"

if REPO_URL and not os.path.exists(WORK_DIR):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {WORK_DIR}
elif not os.path.exists(WORK_DIR):
    raise RuntimeError(
        "REPO_URL is empty and /content/rule-forge does not exist. "
        "Upload a zip and unzip it, or set REPO_URL."
    )

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la data/datasets/toxicn_*.json 2>/dev/null || echo 'Dataset JSONs not found — will need to generate them'

In [ ]:
!pip install -q -r packages/python/requirements.txt

## 3. Verify dataset files exist

If the JSON splits are not in the repo (`.gitignore`), you need to either:
- Upload `toxicn_train.json`, `toxicn_val.json`, `toxicn_test.json` to `data/datasets/`
- Or run the import script (requires Node.js, not available on Colab by default)

The cell below checks and offers a fallback.

In [ ]:
import os, json

REQUIRED = [
    "data/datasets/toxicn_train.json",
    "data/datasets/toxicn_val.json",
    "data/datasets/toxicn_test.json",
]
missing = [f for f in REQUIRED if not os.path.exists(f)]

if missing:
    print("Missing dataset files:", missing)
    print()
    print("Option 1: Upload them via Colab file browser (left panel) into data/datasets/")
    print("Option 2: Run the cell below to upload from your machine")
else:
    for f in REQUIRED:
        d = json.load(open(f, encoding='utf-8'))
        pos = sum(1 for x in d if x['label'] == 1)
        print(f"{f}: {len(d)} samples (toxic={pos}, non-toxic={len(d)-pos})")

In [ ]:
# Run this cell ONLY if dataset files are missing
# It will prompt you to upload the 3 JSON files from your local machine

import os
if any(not os.path.exists(f) for f in REQUIRED):
    from google.colab import files
    os.makedirs("data/datasets", exist_ok=True)
    print("Upload toxicn_train.json, toxicn_val.json, toxicn_test.json")
    uploaded = files.upload()
    for name, content in uploaded.items():
        dest = f"data/datasets/{name}"
        with open(dest, 'wb') as f:
            f.write(content)
        print(f"  Saved {dest} ({len(content)} bytes)")
else:
    print("All dataset files present, skipping upload.")

## 4. Train BERT

Fine-tune `bert-base-chinese` on `toxicn_train.json`, early-stop on val, select threshold via val F1.

Expected time: ~30 min on T4, ~15 min on A100.

In [ ]:
!python packages/python/train_bert.py \
    --input-field content \
    --threshold-mode val_f1 \
    --train-json data/datasets/toxicn_train.json \
    --val-json   data/datasets/toxicn_val.json \
    --out-dir    data/models/bert_base_zh_v1 \
    --epochs 3 \
    --batch-size 16 \
    --max-length 128 \
    --seed 42

## 5. Predict on test set

Uses the frozen `decision_threshold.json` from the training step.

In [ ]:
!python packages/python/predict_bert.py \
    --model-dir data/models/bert_base_zh_v1 \
    --test-json data/datasets/toxicn_test.json

## 6. Inspect outputs

In [ ]:
import json

MODEL_DIR = "data/models/bert_base_zh_v1"

for name in ["metrics_val.json", "decision_threshold.json", "train_config.json"]:
    path = f"{MODEL_DIR}/{name}"
    print(f"\n=== {name} ===")
    print(json.dumps(json.load(open(path)), indent=2))

# Quick test metrics preview
pred_path = f"{MODEL_DIR}/predictions_test.json"
pred = json.load(open(pred_path))
preds = pred["predictions"]
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
y_true = [p["gold_label"] for p in preds]
y_pred = [p["predicted_label"] for p in preds]
print(f"\n=== Test metrics (quick check) ===")
print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  F1:        {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Threshold: {pred['decision_threshold']} ({pred['threshold_source']})")
print(f"  Input:     {pred['bert_input_field']}")
print(f"  Samples:   {len(preds)}")

## 7. Download artifacts

Pack the 3 key files (not the full model weights) for local registration.

In [ ]:
import tarfile, os

MODEL_DIR = "data/models/bert_base_zh_v1"
ARTIFACT = "bert_artifacts.tar.gz"

files_to_pack = [
    f"{MODEL_DIR}/metrics_val.json",
    f"{MODEL_DIR}/train_config.json",
    f"{MODEL_DIR}/decision_threshold.json",
    f"{MODEL_DIR}/predictions_test.json",
]

with tarfile.open(ARTIFACT, "w:gz") as tar:
    for f in files_to_pack:
        tar.add(f)
        print(f"  packed: {f}")

print(f"\nArchive: {ARTIFACT} ({os.path.getsize(ARTIFACT)} bytes)")
print("Download this file, then on your local machine:")
print("  tar xzf bert_artifacts.tar.gz -C /path/to/rule-forge/")
print("  cd /path/to/rule-forge && npm run register:bert")

In [ ]:
from google.colab import files
files.download(ARTIFACT)